In [ ]:
!pip install -U peft bitsandbytes transformers accelerate

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T" # for small memory

In [ ]:
import fitz

def extract_text_from_pdf(pdf_path):
    text_blocks = []
    with fitz.open(pdf_path) as doc:
        for page in doc:
            text = page.get_text("text").strip()
            if text:
                text_blocks.append(text)
    return text_blocks

In [ ]:
text_data = extract_text_from_pdf("LLM Finetuning  02 Introduction of Finetuning.pdf")

In [ ]:
import re
def split_paragraph(pages):
    paragraphs = []
    for page_text in pages:
        # split on double line breaks
        chunks = re.split(r'\n\s\n', page_text)
        for chunk in chunks:
            clean = chunk.strip()
            if len(clean) > 30: # ignore too short lines
                paragraphs.append(clean)
    return paragraphs

In [ ]:
chunks = split_paragraph(pages=text_data)

In [ ]:
data = [{"text": p} for p in chunks]

In [ ]:
from datasets import Dataset
dataset = Dataset.from_list(data)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# preprocess
def tokenize_fn(examples):
    tokens = tokenizer(examples['text'], truncation=True, padding="max_length", max_length=512)
    tokens['labels'] = tokens['input_ids'].copy()
    return tokens

In [ ]:
tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=['text'])

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"": 0}
)

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "v_proj"]
)

In [ ]:
q_lora_model = get_peft_model(model, lora_config)

In [ ]:
training_args = TrainingArguments(
    output_dir="./TinyLama_results",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=2,
    logging_steps=50,
    learning_rate=2e-5,
    fp16=True,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=q_lora_model,
    args=training_args,
    train_dataset=tokenized
)

In [ ]:
trainer.train()

In [ ]:
model_pth = "/content/TinyLama_results/checkpoint-350"

In [ ]:
trained_model = AutoModelForCausalLM.from_pretrained(
    model_pth,
    device_map={"": 0}
)

In [ ]:
prompt = "fine tuning techniques there are many of them"
input = tokenizer(prompt, return_tensors='pt').to('cuda')

In [ ]:
outputs = trained_model.generate(
    **input,
    max_new_tokens=100,
    temperature=0.7,
    do_sample=True,
    top_p=0.95,
    repetition_penalty=1.2
)

In [ ]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### Intsruction FineTuning